<div style="background: linear-gradient(135deg, #118ab2 0%, #073b4c 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🏰 01 — Montando o Bloco Transformer</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 06 · Bloco 03 · Arquitetura</p>
</div>

## 🎯 Objetivos

- Juntar Attention + Feed Forward + Skip Connections
- O que forma um Bloco *Encoder*

---

## 1️⃣ O Bloco Encoder do Transformer

A Atenção sozinha não é forte suficiente. Após a palavra entender seu contexto com a Atenção, ela passa por uma MLP (Multi-Layer Perceptron) individual para processar essa informação.

E como a rede é muito funda, as **Skip Connections** (Residuais, lembra da Fase 05?) são obrigatórias, junto com o **LayerNorm**.

In [ ]:
import torch
import torch.nn as nn

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model=512, heads=8, dim_feedforward=2048):
        super().__init__()
        # 1. Atenção
        self.atencao = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        
        # 2. Feed Forward (MLP)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        
    def forward(self, x):
        # Parte 1: Attention + Skip Connection + Norm
        # Obs: Modelos modernos aplicam Norm ANTES da atenção (Pre-LN)
        x_att, _ = self.atencao(x, x, x)
        x = self.norm1(x + x_att)  # O '+ x' é a Skip Connection!
        
        # Parte 2: FFN + Skip Connection + Norm
        x_ffn = self.ffn(x)
        x = self.norm2(x + x_ffn)
        return x

bloco = TransformerEncoderBlock()
print('Bloco Encoder Criado! O BERT usa 12 desses encadeados.')

## 2️⃣ Testando o Fluxo

Note que o shape **não muda**. Entra `(Batch, Seq, Dim)` e sai `(Batch, Seq, Dim)`. Cada token foi enriquecido com contexto.

In [ ]:
entrada = torch.randn(2, 50, 512) # 2 textos de 50 palavras
saida = bloco(entrada)
print(f'Entrada: {entrada.shape} -> Saída: {saida.shape}')

## 🏋️ Questões

### Q1. O que é LayerNorm e como ele difere do BatchNorm que vimos nas CNNs?

### Q2. Qual a principal diferença estrutural entre um modelo Encoder (como BERT) e um modelo Decoder (como GPT) em relação à máscara de atenção (Masked Attention)?

In [ ]:
# Resolva aqui
